# Решение: поиск ботов по кукам

Итоговый ноутбук: признаки по кукам, обучение CatBoost на всём train, предсказание для test и файл `outputs/submission.csv`. Разведка и эксперименты, из которых взяты признаки и настройки, лежат в `01_eda.ipynb` и `02_experiments.ipynb`.

Ноутбук самодостаточен и читает только `data/`, `metric.py` и `features.py` из корня проекта. Окружение: Python 3.12, версии библиотек в `requirements.txt`.

## 1. Окружение

Библиотеки и фиксированные параметры.
- `SEEDS`: пять сидов, итоговый score усредняется по пяти моделям (устойчивее одной)
- `thread_count = 4`: число потоков зафиксировано, чтобы результат не зависел от машины
- `metric.py` используется только для проверки пайплайна, тем же файлом считает проверяющая система
- признаки собираются функциями из `features.py`, общими с `02_experiments.ipynb`, чтобы код не дублировался между ноутбуками

In [1]:
import platform
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import catboost
from catboost import CatBoostClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_DIR = ROOT / "data"
OUT_PATH = ROOT / "outputs" / "submission.csv"
sys.path.insert(0, str(ROOT))
from metric import precision_at_recall
from features import (
    prepare_events, feats_volume, feats_timing, feats_search, feats_diversity,
    feats_pointer, feats_tech, feats_age, feats_popularity, feats_timing2, feats_cursor2, feats_ua2,
)

SEEDS = [7, 13, 21, 34, 55]
CAT_PARAMS = dict(iterations=500, learning_rate=0.03, depth=6, l2_leaf_reg=3)
POPULARITY_DAYS = 3

print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__, "| catboost", catboost.__version__)

Python 3.12.3 | numpy 1.26.4 | pandas 2.2.3 | catboost 1.2.10


## 2. Данные

In [2]:
DATE_COLUMNS = ["cookie_created_at", "window_start_ts", "window_end_ts"]

train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=DATE_COLUMNS)
test = pd.read_csv(DATA_DIR / "test.csv", parse_dates=DATE_COLUMNS)
events = pd.read_csv(DATA_DIR / "events.csv.gz", parse_dates=["event_ts"])
meta = pd.concat([train.assign(split="train"), test.assign(split="test")], ignore_index=True)

print("train:", train.shape, "| test:", test.shape, "| events:", events.shape)

train: (11091, 5) | test: (4909, 4) | events: (328905, 14)


Подготовка событий.
- удаление дублей
- платформа приведена к web / android / ios
- только события внутри окна куки (`window_start_ts <= event_ts < window_end_ts`)
- сортировка по куке и времени
- проверка, что `captcha_shown` не остался

In [3]:
ev = prepare_events(events, meta)
print("событий в окне:", len(ev), "| кук с событиями:", ev["cookie_id"].nunique(), "из", len(meta))

событий в окне: 283833 | кук с событиями: 16000 из 16000


## 3. Признаки

71 признак на куку: объём событий, тайминг, поиск, разнообразие, курсор, платформа и UA, возраст куки, популярность объявлений — код в `features.py`.

Общая таблица `X`: одна строка на куку (train и test вместе), 71 признак. Порядок признаков зафиксирован.
- число поисков у кук без поиска равно 0
- остальные пропуски остаются, CatBoost работает с ними сам
- категориальные признаки (платформа, тип UA, тип UA подробнее, устройство) передаются модели как строки

In [8]:
parts = [
    feats_volume(ev), feats_timing(ev), feats_search(ev), feats_diversity(ev), feats_pointer(ev), feats_tech(ev), feats_age(meta),
    feats_popularity(ev, K=POPULARITY_DAYS), feats_timing2(ev), feats_cursor2(ev), feats_ua2(ev),
]
FINAL_COLS = [c for p in parts for c in p.columns]
CAT_COLS = ["platform", "ua_kind", "ua_form", "device"]

X = pd.concat(parts, axis=1).reindex(meta["cookie_id"]).reset_index(drop=True)
X["n_search"] = X["n_search"].fillna(0)
X[CAT_COLS] = X[CAT_COLS].astype(str)

y = meta["target"]
is_train = (meta["split"] == "train").values
day = (meta["window_start_ts"] - pd.Timestamp("2026-04-06")).dt.days.values

assert len(FINAL_COLS) == 71 and len(set(FINAL_COLS)) == 71
assert len(X) == len(train) + len(test) and X.index.is_unique
print("признаков:", len(FINAL_COLS), "| кук:", len(X), "| train:", int(is_train.sum()), "| test:", int((~is_train).sum()))

признаков: 71 | кук: 16000 | train: 11091 | test: 4909


## 4. Проверка пайплайна

Функции обучения.
- `fit_ensemble`: пять моделей CatBoost с разными сидами на выбранных строках
- `predict_ensemble`: среднее вероятностей по пяти моделям, значение остаётся в диапазоне 0–1 и почти без повторов

Настройки CatBoost: `iterations` (число деревьев), `learning_rate` (шаг обучения), `depth` (глубина деревьев), `l2_leaf_reg` (штраф за сложность).

In [9]:
def fit_ensemble(idx, seeds=SEEDS):
    models = []
    for seed in seeds:
        model = CatBoostClassifier(**CAT_PARAMS, random_seed=seed, thread_count=4, verbose=0, allow_writing_files=False)
        model.fit(X.iloc[idx][FINAL_COLS], y.iloc[idx].astype(int), cat_features=CAT_COLS)
        models.append(model)
    return models

def predict_ensemble(models, idx):
    return np.mean([m.predict_proba(X.iloc[idx][FINAL_COLS])[:, 1] for m in models], axis=0)

Проверка на отложенных днях: обучение на 06–16.04, проверка на 17–19.04 (то же разбиение, что в quickstart и в экспериментах). Метки этих дней в обучение не попадают. Один сид и среднее по пяти.

In [ ]:
tr_idx = np.where(is_train & (day <= 10))[0]
va_idx = np.where(is_train & (day >= 11))[0]
y_va = y.iloc[va_idx].astype(int).values

models_hold = fit_ensemble(tr_idx)
single = models_hold[0].predict_proba(X.iloc[va_idx][FINAL_COLS])[:, 1]
averaged = predict_ensemble(models_hold, va_idx)

check = pd.DataFrame({
    "один сид": [precision_at_recall(y_va, single), average_precision_score(y_va, single), roc_auc_score(y_va, single)],
    "среднее по 5 сидам": [precision_at_recall(y_va, averaged), average_precision_score(y_va, averaged), roc_auc_score(y_va, averaged)],
}, index=["P@R0.7", "PR-AUC", "ROC-AUC"])
print(f"обучение: {len(tr_idx)} кук, проверка: {len(va_idx)} кук, ботов в проверке: {y_va.sum()}")
display(check.round(3))

обучение: 9140 кук, проверка: 1951 кук, ботов в проверке: 160


,один сид,среднее по 5 сидам
P@R0.7,0.818,0.814
PR-AUC,0.800,0.799
ROC-AUC,0.936,0.937


На отложенных днях P@R0.7 около 0,82 на одном сиде и 0,81 в среднем по пяти — усреднение почти не меняет метрику, разница в пределах шума.

## 5. Финальная модель

Финальное обучение: пять моделей CatBoost на всём train (14 дней, все метки) и предсказание для test. Итоговый score куки: среднее вероятностей по пяти моделям.

In [11]:
train_idx = np.where(is_train)[0]
test_idx = np.where(~is_train)[0]

final_models = fit_ensemble(train_idx)
score = predict_ensemble(final_models, test_idx)

print("моделей:", len(final_models), "| кук в test:", len(test_idx))
print("score: мин", score.min().round(4), "| медиана", np.median(score).round(4), "| макс", score.max().round(4), "| среднее", score.mean().round(4))

моделей: 5 | кук в test: 4909
score: мин 0.0003 | медиана 0.0173 | макс 0.9991 | среднее 0.0834


Средний score 0,083 близок к доле ботов в train (8,1%).

## 6. Файл ответа

Файл `outputs/submission.csv`: ровно две колонки `cookie_id` и `score`, по одной строке на каждую куку из `test.csv`, строки в порядке `test.csv`. Переносы строк `\n`

In [12]:
submission = pd.DataFrame({"cookie_id": meta.loc[~is_train, "cookie_id"].values, "score": score})
OUT_PATH.parent.mkdir(exist_ok=True)
submission.to_csv(OUT_PATH, index=False, lineterminator="\n")
print("записано:", OUT_PATH, "| строк:", len(submission))

записано: /mnt/c/Users/ASUS/Desktop/Avito-Bootcamp/avito-antibot/outputs/submission.csv | строк: 4909


In [13]:
saved = pd.read_csv(OUT_PATH)
sample = pd.read_csv(ROOT.parent / "task" / "sample_submission.csv") if (ROOT.parent / "task" / "sample_submission.csv").exists() else None

assert list(saved.columns) == ["cookie_id", "score"]
assert len(saved) == len(test) and saved["cookie_id"].is_unique
assert set(saved["cookie_id"]) == set(test["cookie_id"])
assert (saved["cookie_id"].values == test["cookie_id"].values).all()
assert saved["score"].dtype == "float64" and np.isfinite(saved["score"]).all() and saved["score"].between(0, 1).all()
if sample is not None:
    assert list(sample.columns) == list(saved.columns) and set(sample["cookie_id"]) == set(saved["cookie_id"])

raw = OUT_PATH.read_bytes()
assert raw.startswith(b"cookie_id,score\n") and b"\r" not in raw and raw.endswith(b"\n")
print("все проверки пройдены")
print("строк:", len(saved), "| разных score:", saved["score"].nunique(), f"({saved['score'].nunique() / len(saved):.1%})")
display(saved.head())

все проверки пройдены
строк: 4909 | разных score: 4909 (100.0%)


,cookie_id,score
0,ck_315fb710a0e371e7,0.001352
1,ck_a76ee3b3e3e522fd,0.183982
2,ck_94c9a4d382689e82,0.022221
3,ck_8eaf9509ad9462a0,0.002518
4,ck_9a88a5a989cb5bc6,0.014964


Файл соответствует требованиям: 4 909 строк, по одной на каждую куку из test, все score разные.

## Итог

- **Результат:** файл `outputs/submission.csv` с колонками `cookie_id` и `score` для всех 4 909 кук из test.
- **Модель:** CatBoost (глубина 6, 500 деревьев, шаг 0,03), 71 признак, пять сидов усредняются.
- **Ожидаемое качество:** по проверке во времени P@R0.7 около 0,79 (baseline из quickstart около 0,10). Оценка шумная, на test возможен разброс.
- **Воспроизводимость:** сиды и число потоков зафиксированы, версии библиотек в `requirements.txt`, все признаки считаются по событиям внутри окна (популярность объявлений по событиям, произошедшим к концу окна).